In [1]:
# General imports
import os
import json
import collections

# Data science imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# import plotly
import scipy.sparse as sp

import torch.nn.functional as F
from torch.utils.data import random_split
import torch
from torch.nn import Linear
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.nn import global_mean_pool

import torch_geometric
from torch_geometric.nn import GCNConv
from torch_geometric.nn import GATConv
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import to_networkx

import networkx as nx
from networkx.algorithms import community

from tqdm.auto import trange
from tqdm import tqdm

import sklearn
from sklearn.model_selection import train_test_split
from torch_geometric.loader import DataLoader
from torch.utils.data import Subset


/Users/smayan/opt/anaconda3/envs/flowback/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
from torch_geometric.datasets import TUDataset
dataset_path = "data/TUDataset"
dataset = TUDataset(root=dataset_path, name='PROTEINS')
dataset.download()

In [4]:
# split dataset into train, test, and unlabeled sets for Deep Active Learning
# 10% test set, 10% initial labeled set, rest unlabeled
total = len(dataset)
n_test = int(0.2* total)
n_init = int(0.1 * total)
n_unlabeled = total - n_test - n_init

# shuffle once before split
perm = torch.randperm(total)

dataset = dataset[perm]  # shuffle

labeled_set = dataset[:n_init]
unlabeled_set = dataset[n_init:n_init+n_unlabeled]
test_set = dataset[n_init+n_unlabeled:]

#to be used for the active learning loop
labeled_indices = np.array(list(range(n_init)))
unlabeled_indices = np.array(list(range(n_init, n_init+n_unlabeled)))

labeled_loader = DataLoader(labeled_set, batch_size=20, shuffle=True)
test_loader = DataLoader(test_set, batch_size=1, shuffle=True)


### Models - GCN, GAT, SAGE - Three Canonical GNN Models

In [5]:
class GCN(torch.nn.Module):
    def __init__(self, hidden_channels):
        super(GCN, self).__init__()
        torch.manual_seed(12345)
        self.conv1 = GCNConv(dataset.num_node_features, hidden_channels)
        # self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.conv3 = GCNConv(hidden_channels, hidden_channels)

        # ok for this dataset
        self.lin = Linear(hidden_channels, dataset.num_classes)

    def forward(self, x, edge_index, batch):
        # 1. Obtain node embeddings
        x = self.conv1(x, edge_index)
        x = x.relu()
        x = self.conv3(x, edge_index)

        # 2. Readout layer
        x = global_mean_pool(x, batch)  # [batch_size, hidden_channels]

        # # 3. Apply a final classifier
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.lin(x)
        return x

class GAT(nn.Module):
    def __init__(self, hidden_channels):
        super(GAT, self).__init__()
        torch.manual_seed(12345)
        self.gat1 = GATConv(dataset.num_node_features, hidden_channels)
        self.gat2 = GATConv(hidden_channels, hidden_channels)

        self.lin = Linear(hidden_channels, dataset.num_classes)

    def forward(self, x, edge_index, batch):

        x = self.gat1(x, edge_index)
        x = x.relu()
        x = self.gat2(x, edge_index)

        #find graph embeddings.
        x = global_mean_pool(x, batch)  # [batch_size, hidden_channels]

        #linear classifier.
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.lin(x)

        return x

class SAGE(torch.nn.Module):
    def __init__(self, hidden_channels):
        super(SAGE, self).__init__()
        torch.manual_seed(12345)
        self.sage1 = SAGEConv(dataset.num_node_features, hidden_channels)
        self.sage2 = SAGEConv(hidden_channels, hidden_channels)

        self.lin = Linear(hidden_channels, dataset.num_classes)

    def forward(self, x, edge_index, batch):

        x = self.sage1(x, edge_index)
        x = x.relu()
        x = self.sage2(x, edge_index)

        #find graph embeddings.
        x = global_mean_pool(x, batch)  # [batch_size, hidden_channels]

        #linear classifier.
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.lin(x)

        return x

### Train & Test functions

In [6]:
def train(model, optimizer, labeled_loader, num_epochs = 10):
    """Training function for the model."""
    model.train()
    total_loss = 0
    for epoch in tqdm(range(num_epochs)):
        for data in labeled_loader:
            data = data.to(device)
            optimizer.zero_grad()
            out = model(data.x, data.edge_index, data.batch)
            loss = F.cross_entropy(out, data.y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * data.num_graphs
    return total_loss / len(labeled_loader.dataset)

def evaluate(model, test_loader):
    """Evaluation function for the model."""
    model.eval()
    correct = 0
    total_loss = 0
    with torch.no_grad():
        # Iterate over the test set
        for data in test_loader:
            data = data.to(device)
            out = model(data.x, data.edge_index, data.batch)
            pred = out.argmax(dim=1)  # Use the class with highest probability.
            loss = F.cross_entropy(out, data.y)
            total_loss += loss.item() * data.num_graphs

            correct += int((pred == data.y).sum())
    return correct / len(test_loader.dataset), total_loss / len(test_loader.dataset)

In [ ]:
model = GAT(hidden_channels=64).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

test_loss = train(model, optimizer, labeled_loader, num_epochs=20)
acc, loss = evaluate(model, test_loader)

  0%|          | 0/20 [00:00<?, ?it/s]

In [ ]:
print(f"Test Accuracy: {acc * 100 :.4f}")
print(f"Test Loss: {loss:.4f}")

Test Accuracy: 61.7117
Test Loss: 0.6594


### Query Strategies

In [ ]:
def random_aquisition(unlabeled_set, k=10, model=None):
    """
    Returns the indices of k random samples from the unlabeled set.
    """
    # Randomly select k samples from the unlabeled set
    indices = np.random.choice(len(unlabeled_set), size=k, replace=False)
    return indices
    

### Active Learning Loop

In [ ]:
model = GAT(hidden_channels=64).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.00001, weight_decay=5e-4)

In [ ]:
labeled_indices_copy = labeled_indices.copy()
labeled_set_copy = labeled_set.copy()
unlabeled_indices_copy = unlabeled_indices.copy()
unlabeled_set_copy = unlabeled_set.copy()
test_set_copy = test_set.copy()

In [ ]:
n_rounds = 10

for i in range(n_rounds):
    print(f"Round {i+1}/{n_rounds}")
    # Train the model on the labeled set
    train_loss = train(model, optimizer, labeled_loader, num_epochs=20)
    acc, loss = evaluate(model, test_loader)
    print(f"Test Accuracy: {acc * 100 :.4f}")
    print(f"Test Loss: {loss:.4f}")

    # Randomly select k samples from the unlabeled set
    indices = random_aquisition(unlabeled_set_copy, k=10)
    # Add the selected indices to the labeled set indices
    labeled_indices_copy = np.concatenate((labeled_indices_copy, unlabeled_set_copy[indices]))
    # Remove the selected indices from the unlabeled set indices
    np.delete(unlabeled_indices_copy, indices)

    #now redefine the labeled and unlabeled sets
    labeled_set_copy = Subset(dataset, labeled_indices_copy)
    print(len(labeled_set))
    unlabeled_set_copy = Subset(dataset, unlabeled_indices_copy)

    # Create a new DataLoader for the updated labeled set
    labeled_loader = DataLoader(labeled_set, batch_size=20, shuffle=True)


Round 1/10


NameError: name 'train' is not defined

In [ ]:
labeled_set[0]

Data(edge_index=[2, 96], x=[22, 3], y=[1])